# HW05 — Benchmark ASR en GPU (Colab)

Este notebook **transcribe** todos los audios con cada combinación de la grilla
(`experimentos` en `config.yaml`) y guarda cada transcript en el cache.
Es el **único** lugar donde se transcribe: la app y GitHub Actions solo leen el cache.

**Antes de empezar**
1. *Entorno de ejecución → Cambiar tipo de entorno → GPU* (T4 alcanza; la grilla del config
   está pensada para terminar en menos de 40 min con ~8 clips de 2–4 min).
2. Sube los audios a Google Drive con esta estructura (los nombres de archivo = id del clip):
```
MyDrive/HW05/audio/
    entrevistas/  E01.m4a  E02.m4a  E03.m4a  E04.m4a
    llamadas/     L01.m4a  L02.m4a  L03.m4a  L04.m4a
    ruido/        ambiente.wav      <- ruido ambiente grabado para L3
```
(cualquier extensión de `degradacion.extensiones_audio` sirve: .m4a, .wav, .mp3, .ogg...)

**Qué produce**: `MyDrive/HW05/cache/` con los transcripts (`.json`), el SNR medido
de la escalera y una tabla de corridas. Al final se descarga un `.zip` para copiarlo al repo.

Si Colab se desconecta, vuelve a correr todo: lo que ya está en el cache **no se re-transcribe**.

In [ ]:
# 1. ¿Hay GPU?
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 2. Drive y repo
from google.colab import drive
drive.mount('/content/drive')

CARPETA_DRIVE = '/content/drive/MyDrive/HW05'   # <- cambia si usaste otra carpeta
REPO = 'https://github.com/JazminGalindoo/HW05_VOZ_A_REGISTRO.git'

%cd /content
!rm -rf HW05_VOZ_A_REGISTRO
!git clone -q $REPO
%cd /content/HW05_VOZ_A_REGISTRO
!git log --oneline -1

In [ ]:
# 3. Dependencias con las versiones fijadas del repo.
#    Si Colab pide "Restart session", reinicia y vuelve a correr desde la celda 2.
!pip install -q -r requirements.txt

### Arreglo CUDA 12 (Colab trae CUDA 13)
CTranslate2 4.8.4 (motor de faster-whisper) está compilado para **CUDA 12 + cuDNN 9**, pero Colab
ahora trae CUDA 13: al transcribir en GPU falla con `Library libcublas.so.12 is not found`.
Se instalan las librerías de CUDA 12 como paquetes de pip y se **precargan** con `ctypes`
antes de importar faster-whisper, para que CTranslate2 las encuentre.

**Si reinicias la sesión, vuelve a correr esta celda antes que las demás.**

In [ ]:
# 3b. Librerías CUDA 12 para CTranslate2 + prueba real en GPU
!pip install -q nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"

import ctypes, glob, os, site

libs = []
for carpeta in site.getsitepackages():
    for paquete in ['cublas', 'cudnn']:
        libs += sorted(glob.glob(os.path.join(carpeta, 'nvidia', paquete, 'lib', 'lib*.so*')))
assert any('libcublas.so.12' in l for l in libs), 'No se instaló libcublas.so.12'

# RTLD_GLOBAL: los símbolos quedan visibles para CTranslate2. Algunas librerías
# dependen de otras, así que se reintenta hasta que carguen todas.
pendientes = libs
for _ in range(3):
    fallan = []
    for so in pendientes:
        try:
            ctypes.CDLL(so, mode=ctypes.RTLD_GLOBAL)
        except OSError:
            fallan.append(so)
    pendientes = fallan
print(f'{len(libs) - len(pendientes)}/{len(libs)} librerías CUDA 12 precargadas')

# Prueba: modelo tiny en GPU transcribiendo 1 s de silencio. Si esto falla, la grilla también fallaría.
import numpy as np
from faster_whisper import WhisperModel
prueba = WhisperModel('tiny', device='cuda', compute_type='float16', download_root='models')
list(prueba.transcribe(np.zeros(16000, dtype=np.float32), language='es')[0])
print('OK: CTranslate2 transcribe en', prueba.model.device)
del prueba

In [ ]:
# 4. Los audios de Drive aparecen dentro del repo en data/audio (enlace simbólico),
#    así las rutas de config.yaml funcionan igual que en local.
import os
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/audio'):
    os.symlink(f'{CARPETA_DRIVE}/audio', 'data/audio')
!ls data/audio/entrevistas data/audio/llamadas data/audio/ruido

In [ ]:
# 5. Versiones y prueba de GPU (leída de CTranslate2, no de config)
import inspect
import ctranslate2, faster_whisper
from faster_whisper import WhisperModel

print('faster-whisper', faster_whisper.__version__, '| ctranslate2', ctranslate2.__version__)
print('GPUs que ve CTranslate2:', ctranslate2.get_cuda_device_count())
print('¿transcribe() tiene hotwords?', 'hotwords' in inspect.signature(WhisperModel.transcribe).parameters)
assert ctranslate2.get_cuda_device_count() > 0, 'Sin GPU: Entorno de ejecución > Cambiar tipo > GPU'

## Escalera de degradación (Tarea 2)
Crea L1 (WhatsApp), L2 (teléfono) y L3 (L1 + ruido) de cada llamada en
`MyDrive/HW05/audio/degradado/` y escribe el **SNR medido** en `cache/results/degradacion.json`.

In [ ]:
# 6. Degradar las llamadas (ffmpeg ya viene instalado en Colab)
!python -m src.degrade

## Plan de corridas
Combinaciones L0 sobre todos los clips + escalera L1–L3 sobre las llamadas.
El cache se guarda **directo en Drive** para no perder nada si Colab se desconecta.

In [ ]:
# 7. Lista de transcripciones a hacer
from pathlib import Path
from src.engine import cargar_config, con_motor, process_audio
from src import asr

cfg = cargar_config()
cfg['rutas']['cache_transcripts'] = f'{CARPETA_DRIVE}/cache/transcripts'   # persistente
R, D, E = cfg['rutas'], cfg['degradacion'], cfg['experimentos']


def audios(carpeta):
    return sorted(p for p in Path(carpeta).iterdir() if p.suffix.lower() in D['extensiones_audio'])


def ruta_nivel(clip, nivel):
    return Path(R['audio_degradado']) / nivel / f"{clip}{D[nivel]['extension']}"


trabajos = []   # (audio, schema, opciones de motor)
for schema, carpeta in [('entrevista', R['audio_entrevistas']), ('pedido', R['audio_llamadas'])]:
    for audio in audios(carpeta):
        for combo in E['combinaciones_l0']:
            # "clips" (opcional) limita una combinación a algunos clips (p. ej. CPU: es lenta)
            if 'clips' in combo and audio.stem not in combo['clips']:
                continue
            opciones = {k: v for k, v in combo.items() if k != 'clips'}
            trabajos.append((audio, schema, dict(opciones, nivel='L0')))
for audio in audios(R['audio_llamadas']):
    for nivel in E['escalera']['niveles']:
        for combo in E['escalera']['combinaciones']:
            trabajos.append((ruta_nivel(audio.stem, nivel), 'pedido', dict(combo, nivel=nivel)))

# Ordenar por modelo: cada modelo se carga una sola vez
trabajos.sort(key=lambda t: (t[2]['modelo'], t[2]['device'], t[2]['compute_type']))
faltan = [t for t in trabajos if not asr.ruta_cache(t[0].stem, con_motor(cfg, **t[2])).exists()]
print(f'{len(trabajos)} transcripciones en la grilla; faltan {len(faltan)}')

In [ ]:
# 8. Correr la grilla (lo que ya está en cache se salta)
import time

resumen, modelo_actual = [], None
for i, (audio, schema, opciones) in enumerate(trabajos, 1):
    clave = (opciones['modelo'], opciones['device'], opciones['compute_type'])
    if clave != modelo_actual:
        asr.liberar_modelos()           # soltar la VRAM del modelo anterior
        modelo_actual = clave
        pendiente = any(t[2]['modelo'] == clave[0] and t[2]['device'] == clave[1]
                        and t[2]['compute_type'] == clave[2] for t in faltan)
        if E['calentar'] and pendiente:
            # Corrida descartada: la primera incluye inicializar CUDA y cargar pesos
            tmp = con_motor(cfg, **opciones, usar_cache=False)
            tmp['rutas']['cache_transcripts'] = '/tmp/calentamiento'
            t0 = time.time(); process_audio(str(audio), schema, tmp)
            print(f'   (calentamiento {clave}: {time.time() - t0:.1f} s)')

    r = process_audio(str(audio), schema, con_motor(cfg, **opciones))
    resumen.append({'clip': audio.stem, 'schema': schema, **opciones,
                    'device_real': r.device, 'compute_real': r.compute_type,
                    'elapsed_s': r.elapsed_s, 'peak_vram_mb': r.peak_vram_mb,
                    'desde_cache': r.desde_cache, 'error': r.error})
    estado = 'cache' if r.desde_cache else f'{r.elapsed_s:.1f} s, VRAM {r.peak_vram_mb} MB'
    print(f'[{i}/{len(trabajos)}] {audio.stem} {opciones} -> {r.device}/{r.compute_type} ({estado}) {r.error or ""}')

In [ ]:
# 9. Tabla de corridas + chequeos
import pandas as pd

df = pd.DataFrame(resumen)
assert df['error'].isna().all(), df[df['error'].notna()][['clip', 'modelo', 'nivel', 'error']]
# Prueba de GPU: lo pedido como cuda corrió en cuda según CTranslate2
assert (df.loc[df['device'] == 'cuda', 'device_real'] == 'cuda').all()

os.makedirs(f'{CARPETA_DRIVE}/cache/results', exist_ok=True)
df.to_csv(f'{CARPETA_DRIVE}/cache/results/corridas_asr.csv', index=False)
df.groupby(['modelo', 'device_real', 'compute_real', 'glosario', 'nivel'])[['elapsed_s', 'peak_vram_mb']].mean().round(2)

In [ ]:
# 10. Llevar el cache al repo: zip para descargar
!cp cache/results/degradacion.json {CARPETA_DRIVE}/cache/results/
!cd {CARPETA_DRIVE} && zip -qr /content/cache_hw05.zip cache
from google.colab import files
files.download('/content/cache_hw05.zip')

## Después, en tu PC
1. Descomprime `cache_hw05.zip` **en la raíz del repo** (quedan `cache/transcripts/*.json` y `cache/results/*`).
2. `pytest` y revisa que todo pase.
3. Commit en una rama nueva (p. ej. `feat/cache-gpu`) y PR. Los audios **no** se suben: están en `.gitignore`.